In [ ]:
# ==========================================
# INSTALL LIBRARIES & PREPARE DATA
# ==========================================
!pip install -q transformers torch scikit-learn pandas matplotlib seaborn tqdm
!cp "/content/drive/MyDrive/LVTN: AI code detection/continue/basic_cpp_ai_detection_dataset.jsonl" "/content/basic_cpp_ai_detection_dataset.jsonl"
file_path = "/content/basic_cpp_ai_detection_dataset.jsonl"

with open(file_path, "r", encoding="utf-8") as f:
    line_count = sum(1 for _ in f)

print("Số dòng trong file:", line_count)

Số dòng trong file: 2850


In [ ]:
import pandas as pd
import json

file_path = "/content/basic_cpp_ai_detection_dataset.jsonl"

data = []
with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        try:
            data.append(json.loads(line))
        except:
            continue

df = pd.DataFrame(data)

# Áp dụng logic tạo Group
df['project_group'] = df['file_name'].apply(lambda x: "_".join(x.split('_')[:2]))

# Đếm tổng quan
n_samples = len(df)
n_groups = df['project_group'].nunique()

print("="*40)
print(f"📊 BÁO CÁO PHÂN TÍCH GROUP (PROJECTS)")
print("="*40)
print(f"🔹 Tổng số file code : {n_samples}")
print(f"🔹 Tổng số Group     : {n_groups}")
print(f"🔹 Trung bình file/Group : {n_samples / n_groups:.1f} files")
print("-" * 40)

# Kịch bản chia Fold
if n_groups >= 10:
    print("✅ ĐÁNH GIÁ: Quá tuyệt vời! Đủ sức chạy 10-Fold CV một cách mượt mà và chuẩn học thuật.")
elif n_groups >= 5:
    print(f"⚠️ ĐÁNH GIÁ: Số group ở mức trung bình ({n_groups}). Nên đổi n_folds = {n_groups} (Tương đương LOGO) để an toàn.")
else:
    print(f"🚨 CẢNH BÁO: Số group QUÁ ÍT ({n_groups}). Có nguy cơ Overfit cao do không đủ sự đa dạng project.")

print("\n📈 TOP 10 GROUP CHIẾM NHIỀU DATA NHẤT:")
print(df['project_group'].value_counts().head(10))

📊 BÁO CÁO PHÂN TÍCH GROUP (PROJECTS)
🔹 Tổng số file code : 2850
🔹 Tổng số Group     : 423
🔹 Trung bình file/Group : 6.7 files
----------------------------------------
✅ ĐÁNH GIÁ: Quá tuyệt vời! Đủ sức chạy 10-Fold CV một cách mượt mà và chuẩn học thuật.

📈 TOP 10 GROUP CHIẾM NHIỀU DATA NHẤT:
project_group
ankitsablok89_Stanford                          20
ecwu_COMP1013                                   20
zhouguanhong_cse100                             10
0xuye0_XYLearn                                  10
Valkorchik_Procedural-Programming-              10
varunk08_algorithms                             10
varunkapoor_Algorithms-and-Data-Structures      10
trivedipankaj_Data-Structures-and-Algorithms    10
trongtn2110_NMLT                                10
TrueElement_CPlusPlus                           10
Name: count, dtype: int64


In [ ]:


# ==========================================
# IMPORTS & CONFIGURATION
# ==========================================
import os
import gc
import json
import random
import shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_cosine_schedule_with_warmup,
    logging
)
from torch.optim import AdamW
from torch.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedGroupKFold, LeaveOneGroupOut # ✅ Thêm LOGO
from sklearn.metrics import (
    classification_report, f1_score,
    precision_recall_curve, accuracy_score
)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

logging.set_verbosity_error()

# ==========================================
# HUGGING FACE LOGIN
# ==========================================
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('colab')
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("✅ Đã đăng nhập Hugging Face thành công!")
except Exception as e:
    print(f"⚠️ Không thể đăng nhập Hugging Face: {e}")

# ==========================================
# CONFIGURATION — Tối ưu cho A100 80GB
# ==========================================
config = {
    "data_path": "/content/basic_cpp_ai_detection_dataset.jsonl",
    "model_name": "microsoft/graphcodebert-base",
    "save_root": "./cv_models",
    "max_len": 512,

    # ✅ CHIẾN THUẬT CROSS-VALIDATION
    # Chọn '10-fold' (khuyên dùng) hoặc 'LOGO' (Leave-One-Group-Out / LOO an toàn)
    "cv_strategy": "10-fold",
    "n_folds": 10, # Chỉ có tác dụng nếu cv_strategy = '10-fold'

    "batch_size": 128,
    "gradient_accumulation_steps": 2,
    "epochs": 5,
    "lr": 3e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "seed": 42,
    "label_smoothing": 0.1,
    "stride": 256,
    "temperature": 2.0,
    "use_bf16": True,
    "gradient_checkpointing": False,
    "num_workers": 4,
    "pin_memory": True,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(config['seed'])

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB")
    print(f"   Strategy: {config['cv_strategy']} | BF16: {config['use_bf16']}")
else:
    print("⚠️ Không tìm thấy GPU!")


⚠️ Không thể đăng nhập Hugging Face: Secret colab does not exist.
✅ GPU: NVIDIA A100-SXM4-40GB | VRAM: 42.4 GB
   Strategy: 10-fold | BF16: True


In [ ]:

# ==========================================
# 1. LOAD & PREPROCESS
# ==========================================
def load_and_preprocess_data(file_path):
    data = []
    if not os.path.exists(file_path):
        print("⚠️ File không tồn tại! Tạo dữ liệu giả lập...")
        for i in range(200):
            data.append({
                "code": f"void func_{i}() {{ int x = {i}; }} " * (i % 5 + 1),
                "label": "AI" if i % 2 == 0 else "Human",
                "file_name": f"user{i//20}_repo{i//10}_task_{i}.cpp"
            })
        df = pd.DataFrame(data)
    else:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                try:
                    data.append(json.loads(line))
                except:
                    continue
        df = pd.DataFrame(data)

    df = df.drop_duplicates(subset=['code'], keep='first').reset_index(drop=True)
    df['label_upper'] = df['label'].astype(str).str.upper()
    df['label_id'] = df['label_upper'].map({"HUMAN": 0, "AI": 1})
    df = df.dropna(subset=['label_id']).reset_index(drop=True)
    df['label_id'] = df['label_id'].astype(int)
    # Gom nhóm theo user_repo để tránh leakage
    df['project_group'] = df['file_name'].apply(lambda x: "_".join(x.split('_')[:2]))
    return df

# ==========================================
# 2. SLIDING WINDOW
# ==========================================
def expand_dataset_sliding_window(df, tokenizer, max_len=510, stride=256):
    new_data = []
    all_texts = df['code'].tolist()
    all_labels = df['label_id'].tolist()

    for text, label in tqdm(zip(all_texts, all_labels), total=len(all_texts), desc="Expanding", leave=False):
        tokens = tokenizer.tokenize(text)
        if len(tokens) <= max_len:
            new_data.append({"code_tokens": tokens, "label_id": label})
        else:
            for i in range(0, len(tokens), stride):
                chunk = tokens[i : i + max_len]
                new_data.append({"code_tokens": chunk, "label_id": label})
                if i + max_len >= len(tokens):
                    break

    return pd.DataFrame(new_data)



In [ ]:
# ==========================================
# 3. DATA SPLIT (Test Hold-out)
# ==========================================
def print_class_distribution(df, name="Dataset"):
    counts = df['label_id'].value_counts()
    ai    = counts.get(1, 0)
    human = counts.get(0, 0)
    total = len(df)
    print(f"   📊 {name}: Tổng {total} | AI: {ai} ({ai/total:.1%}) | HUMAN: {human} ({human/total:.1%})")

def prepare_kfold_data(df, n_trials=30):
    n_groups = df['project_group'].nunique()
    # Nếu dùng LOGO, ta giữ tập Test nhỏ lại một chút để dành data cho CV
    n_splits_test = min(10, n_groups) if config['cv_strategy'] == '10-fold' else min(5, n_groups)

    best_score, best_splits = float('inf'), None
    for seed in range(n_trials):
        sgkf = StratifiedGroupKFold(n_splits=n_splits_test, shuffle=True, random_state=seed)
        train_idx, test_idx = next(sgkf.split(df, df['label_id'], groups=df['project_group']))
        diff = abs(df.iloc[train_idx]['label_id'].mean() - df.iloc[test_idx]['label_id'].mean())
        if diff < best_score:
            best_score, best_splits = diff, (train_idx, test_idx)

    train_idx, test_idx = best_splits
    X_full_train = df.iloc[train_idx].reset_index(drop=True)
    X_test       = df.iloc[test_idx].reset_index(drop=True)

    print("\n✅ Tách Test Hold-out xong:")
    print_class_distribution(X_full_train, "CV Train Pool")
    print_class_distribution(X_test, "Final Test Set")
    return X_full_train, X_test



In [ ]:
# ==========================================
# 4. DATASET & COLLATOR
# ==========================================
tokenizer = RobertaTokenizer.from_pretrained(config['model_name'])

class ExpandedCodeDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.labels = df['label_id'].tolist()
        self.input_ids_list = []
        cls_id = tokenizer.convert_tokens_to_ids(tokenizer.cls_token)
        sep_id = tokenizer.convert_tokens_to_ids(tokenizer.sep_token)

        for tokens in df['code_tokens']:
            ids = [cls_id] + tokenizer.convert_tokens_to_ids(tokens) + [sep_id]
            self.input_ids_list.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': torch.tensor(self.input_ids_list[idx], dtype=torch.long),
            'label':     torch.tensor(self.labels[idx],         dtype=torch.long)
        }

def collate_fn(batch):
    input_ids = [item['input_ids'] for item in batch]
    labels    = [item['label']     for item in batch]
    padded = tokenizer.pad({'input_ids': input_ids}, padding=True, return_tensors="pt")
    return {
        'input_ids':      padded['input_ids'],
        'attention_mask': padded['attention_mask'],
        'labels':         torch.tensor(labels, dtype=torch.long)
    }

# ==========================================
# 5. TRAIN / EVAL FUNCTIONS
# ==========================================
def train_epoch(model, loader, optimizer, scheduler, loss_fn, scaler, accumulation_steps):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="   Train", leave=False)):
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask      = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels    = batch['labels'].to(DEVICE, non_blocking=True)

        if config['use_bf16']:
            with autocast("cuda", dtype=torch.bfloat16):
                outputs = model(input_ids.clone(), attention_mask=mask)
                loss = loss_fn(outputs.logits, labels) / accumulation_steps
            loss.backward()
        else:
            outputs = model(input_ids, attention_mask=mask)
            loss = loss_fn(outputs.logits, labels) / accumulation_steps
            scaler.scale(loss).backward()

        total_loss += loss.item() * accumulation_steps

        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(loader):
            if config['use_bf16']:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            else:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            scheduler.step()
            optimizer.zero_grad()
    return total_loss / len(loader)

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask      = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels    = batch['labels'].to(DEVICE, non_blocking=True)

        if config['use_bf16']:
            with autocast("cuda", dtype=torch.bfloat16):
                outputs = model(input_ids, attention_mask=mask)
        else:
            outputs = model(input_ids, attention_mask=mask)

        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    micro_f1 = f1_score(all_labels, all_preds, average='micro')
    acc = accuracy_score(all_labels, all_preds)
    return macro_f1, micro_f1, acc, all_labels, all_preds

@torch.no_grad()
def eval_epoch_train(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE, non_blocking=True)
        mask      = batch['attention_mask'].to(DEVICE, non_blocking=True)
        labels    = batch['labels'].to(DEVICE, non_blocking=True)
        if config['use_bf16']:
            with autocast("cuda", dtype=torch.bfloat16):
                outputs = model(input_ids, attention_mask=mask)
        else:
            outputs = model(input_ids, attention_mask=mask)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return f1_score(all_labels, all_preds, average='macro'), f1_score(all_labels, all_preds, average='micro')



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [ ]:
# ==========================================
# 6. INFERENCE LOGIC
# ==========================================
@torch.no_grad()
def sliding_window_inference(model, tokenizer, code_text, max_len=510, stride=256, temperature=2.0):
    tokens = tokenizer.tokenize(code_text)
    chunks = []
    if len(tokens) <= max_len:
        chunks.append(tokens)
    else:
        for i in range(0, len(tokens), stride):
            chunks.append(tokens[i : i + max_len])
            if i + max_len >= len(tokens):
                break

    all_ids = []
    for chunk in chunks:
        ids = tokenizer.convert_tokens_to_ids([tokenizer.cls_token] + chunk + [tokenizer.sep_token])
        all_ids.append(ids)

    padded = tokenizer.pad({'input_ids': all_ids}, padding=True, return_tensors="pt")
    input_ids = padded['input_ids'].to(DEVICE)
    mask      = padded['attention_mask'].to(DEVICE)

    if config['use_bf16']:
        with autocast("cuda", dtype=torch.bfloat16):
            outputs = model(input_ids, attention_mask=mask)
    else:
        outputs = model(input_ids, attention_mask=mask)

    probs = torch.softmax(outputs.logits / temperature, dim=1)[:, 1]
    return float(probs.mean().cpu().item())



In [ ]:
# ==========================================
# 7. CHUẨN BỊ CROSS VALIDATION
# ==========================================
df_clean = load_and_preprocess_data(config['data_path'])
X_full_train, X_test = prepare_kfold_data(df_clean)

if os.path.exists(config['save_root']):
    shutil.rmtree(config['save_root'])
os.makedirs(config['save_root'])

loss_fn = nn.CrossEntropyLoss(label_smoothing=config['label_smoothing'])
scaler  = GradScaler("cuda") if not config['use_bf16'] else None

# ✅ LOGIC CHỌN CHIẾN THUẬT CV
if config['cv_strategy'] == 'LOGO':
    print(f"\n🚀 KÍCH HOẠT CHẾ ĐỘ LEAVE-ONE-GROUP-OUT (LOO)")
    cv_splitter = LeaveOneGroupOut()
    splits = list(cv_splitter.split(X_full_train, X_full_train['label_id'], groups=X_full_train['project_group']))
    total_actual_folds = cv_splitter.get_n_splits(groups=X_full_train['project_group'])
    print(f"⚠️ Dữ liệu của bạn có {total_actual_folds} groups. Sẽ tiến hành train {total_actual_folds} folds liên tục!")
else:
    print(f"\n🚀 KÍCH HOẠT CHẾ ĐỘ {config['n_folds']}-FOLD CROSS VALIDATION")
    cv_splitter = StratifiedGroupKFold(n_splits=config['n_folds'], shuffle=True, random_state=config['seed'])
    splits = list(cv_splitter.split(X_full_train, X_full_train['label_id'], groups=X_full_train['project_group']))
    total_actual_folds = config['n_folds']

oof_labels, oof_probs = [], []

print("="*50)



✅ Tách Test Hold-out xong:
   📊 CV Train Pool: Tổng 2512 | AI: 1258 (50.1%) | HUMAN: 1254 (49.9%)
   📊 Final Test Set: Tổng 333 | AI: 167 (50.2%) | HUMAN: 166 (49.8%)

🚀 KÍCH HOẠT CHẾ ĐỘ 10-FOLD CROSS VALIDATION


In [ ]:

# ==========================================
# 8. CROSS VALIDATION TRAINING LOOP
# ==========================================
for fold, (train_idx, val_idx) in enumerate(splits):
    print(f"\n{'='*40}")
    print(f"📁 FOLD {fold+1}/{total_actual_folds}")

    fold_train_files = X_full_train.iloc[train_idx].reset_index(drop=True)
    fold_val_files   = X_full_train.iloc[val_idx].reset_index(drop=True)

    fold_train_expanded = expand_dataset_sliding_window(fold_train_files, tokenizer, stride=config['stride'])
    fold_val_expanded = expand_dataset_sliding_window(fold_val_files, tokenizer, stride=config['stride'])

    train_dataset = ExpandedCodeDataset(fold_train_expanded, tokenizer, config['max_len'])
    val_dataset   = ExpandedCodeDataset(fold_val_expanded,   tokenizer, config['max_len'])

    train_loader = DataLoader(
        train_dataset, batch_size=config['batch_size'], shuffle=True,
        collate_fn=collate_fn, num_workers=config['num_workers'],
        pin_memory=config['pin_memory'], persistent_workers=True, prefetch_factor=2,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=config['batch_size'] * 2, collate_fn=collate_fn,
        num_workers=config['num_workers'], pin_memory=config['pin_memory'],
        persistent_workers=True, prefetch_factor=2,
    )

    model = RobertaForSequenceClassification.from_pretrained(config['model_name'], num_labels=2)
    if config.get('gradient_checkpointing', False):
        model.gradient_checkpointing_enable()
    model.to(DEVICE)

    try:
        model = torch.compile(model, mode="default", dynamic=True)
    except Exception:
        pass

    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": config['weight_decay']},
        {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=config['lr'])

    total_steps  = (len(train_loader) // config['gradient_accumulation_steps']) * config['epochs']
    warmup_steps = int(total_steps * config['warmup_ratio'])
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    best_f1 = 0
    fold_save_path = os.path.join(config['save_root'], f"fold_{fold+1}")

    for epoch in range(config['epochs']):
        train_loss = train_epoch(model, train_loader, optimizer, scheduler, loss_fn, scaler, config['gradient_accumulation_steps'])
        train_macro, _ = eval_epoch_train(model, train_loader)
        val_macro, _, _, _, _ = eval_epoch(model, val_loader)

        macro_gap = train_macro - val_macro
        print(f"   Epoch {epoch+1}: Loss={train_loss:.4f} | Train Macro={train_macro:.4f} | Val Macro={val_macro:.4f} | GAP={macro_gap:+.4f}")

        if macro_gap > 0.05:
            print(f"   ⚠️ CẢNH BÁO: Gap Macro F1 = {macro_gap:.4f} > 0.05 — Dấu hiệu OVERFIT!")

        if val_macro > best_f1:
            best_f1 = val_macro
            os.makedirs(fold_save_path, exist_ok=True)
            unwrapped = model._orig_mod if hasattr(model, '_orig_mod') else model
            unwrapped.save_pretrained(fold_save_path)
            tokenizer.save_pretrained(fold_save_path)

    print(f"✅ Fold {fold+1} Done. Best Val Macro: {best_f1:.4f}")

    # OOF Predictions
    best_model = RobertaForSequenceClassification.from_pretrained(fold_save_path).to(DEVICE)
    best_model.eval()

    fold_oof_probs = []
    for _, row in tqdm(fold_val_files.iterrows(), total=len(fold_val_files), desc="   OOF Inference", leave=False):
        prob = sliding_window_inference(best_model, tokenizer, row['code'], max_len=510, stride=config['stride'], temperature=config['temperature'])
        fold_oof_probs.append(prob)

    oof_probs.extend(fold_oof_probs)
    oof_labels.extend(fold_val_files['label_id'].tolist())

    del model, best_model, optimizer, train_dataset, val_dataset, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()




📁 FOLD 1/10


Expanding:   0%|          | 0/2298 [00:00<?, ?it/s]

Expanding:   0%|          | 0/214 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/539 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Train:   0%|          | 0/94 [00:00<?, ?it/s]

   Epoch 5: Loss=0.2106 | Train Macro=0.9707 | Val Macro=0.9235 | GAP=+0.0472
✅ Fold 10 Done. Best Val Macro: 0.9271


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

   OOF Inference:   0%|          | 0/228 [00:00<?, ?it/s]

In [ ]:

# ==========================================
# 9. TÌM OPTIMAL THRESHOLD BẰNG OOF
# ==========================================
print("\n" + "="*40)
print("🔍 TỐI ƯU HÓA THRESHOLD (Dựa trên OOF Data)")
print("="*40)

precisions, recalls, thresholds = precision_recall_curve(oof_labels, oof_probs)
f1_scores  = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx   = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"🎯 Threshold tối ưu: {best_threshold:.4f} | Max OOF F1: {f1_scores[best_idx]:.4f}")
print(f"💡 Chỉ số OOF F1 ({f1_scores[best_idx]:.4f}) là thước đo độ tin cậy thực tế nhất cho báo cáo khoa học của bạn.")




🔍 TỐI ƯU HÓA THRESHOLD (Dựa trên OOF Data)
🎯 Threshold tối ưu: 0.6523 | Max OOF F1: 0.9776
💡 Chỉ số OOF F1 (0.9776) là thước đo độ tin cậy thực tế nhất cho báo cáo khoa học của bạn.


In [ ]:
# ==========================================
# 10. FINAL TEST TRÊN TẬP HOLD-OUT BẰNG ENSEMBLE
# ==========================================
def predict_ensemble_final(df_test, fold_dirs, tokenizer, threshold):
    print(f"\n🔮 Ensemble Test | Threshold = {threshold:.4f}")
    total_probs = np.zeros(len(df_test))

    for fold_dir in fold_dirs:
        model = RobertaForSequenceClassification.from_pretrained(fold_dir).to(DEVICE).eval()
        fold_probs = []
        for _, row in tqdm(df_test.iterrows(), total=len(df_test), leave=False):
            prob = sliding_window_inference(model, tokenizer, row['code'], max_len=510, stride=config['stride'], temperature=config['temperature'])
            fold_probs.append(prob)
        total_probs += np.array(fold_probs)
        del model
        gc.collect()
        torch.cuda.empty_cache()

    avg_probs   = total_probs / len(fold_dirs)
    predictions = (avg_probs >= threshold).astype(int)
    return predictions, avg_probs

fold_dirs = [os.path.join(config['save_root'], f"fold_{i+1}") for i in range(total_actual_folds)]
y_pred, avg_probs = predict_ensemble_final(X_test, fold_dirs, tokenizer, best_threshold)
y_true = X_test['label_id'].values

print("\n" + "="*50)
print(f"🏆 FINAL REPORT ({config['cv_strategy']} ENSEMBLE TEST SET)")
print("="*50)
print_class_distribution(X_test, "Test Set")
print(classification_report(y_true, y_pred, target_names=['HUMAN', 'AI']))

# ==========================================
# 11. BACKUP LÊN DRIVE
# ==========================================
from google.colab import drive
if not os.path.exists('/content/drive'): drive.mount('/content/drive')

DRIVE_SAVE_PATH = "/content/drive/MyDrive/My_AI_Models/C++_Basic_detection_10_fold_LeaveOneGroupOut_new"
if os.path.exists(DRIVE_SAVE_PATH): shutil.rmtree(DRIVE_SAVE_PATH)
shutil.copytree(config['save_root'], DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)

threshold_info = {"best_threshold": float(best_threshold), "cv_strategy": config['cv_strategy']}
with open(f"{DRIVE_SAVE_PATH}/threshold.json", "w") as f: json.dump(threshold_info, f)

print(f"✅ Đã backup lên: {DRIVE_SAVE_PATH}")


🔮 Ensemble Test | Threshold = 0.6523


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/333 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/333 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/333 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  0%|          | 0/333 [00:00<?, ?it/s]


🏆 FINAL REPORT (10-fold ENSEMBLE TEST SET)
   📊 Test Set: Tổng 333 | AI: 167 (50.2%) | HUMAN: 166 (49.8%)
              precision    recall  f1-score   support

       HUMAN       0.99      0.98      0.99       166
          AI       0.98      0.99      0.99       167

    accuracy                           0.99       333
   macro avg       0.99      0.99      0.99       333
weighted avg       0.99      0.99      0.99       333

✅ Đã backup lên: /content/drive/MyDrive/My_AI_Models/C++_Basic_detection_10_fold_LeaveOneGroupOut_new


### 📂 Metadata của Bộ Dữ liệu (Dataset Metadata)

Dựa trên phân tích từ code, đây là thông tin chi tiết về bộ dữ liệu đang được sử dụng:

*   **Tên file:** `basic_cpp_ai_detection_dataset.jsonl`
*   **Định dạng:** JSON Lines (.jsonl)
*   **Tổng số mẫu (Files):** 2850 mẫu code C++.
*   **Số lượng Nhóm (Project Groups):** 423 nhóm (được trích xuất từ `file_name` theo logic `user_repo`).
*   **Phân bổ Nhãn (Labels):**
    *   `HUMAN` (0): ~50% (Code do người viết).
    *   `AI` (1): ~50% (Code do AI tạo ra).
*   **Đặc điểm dữ liệu:**
    *   **Trường dữ liệu chính:** `code` (nội dung mã nguồn), `label` (nhãn văn bản), `file_name` (tên file chứa thông tin định danh).
    *   **Logic phân tách Group:** `df['file_name'].apply(lambda x: "_".join(x.split('_')[:2]))` dùng để nhóm các file thuộc cùng một tác giả/project.
*   **Chiến lược Cross-Validation:**
    *   **Phương pháp:** Stratified Group K-Fold (10-Fold).
    *   **Mục tiêu:** Đảm bảo toàn bộ code từ một project group chỉ nằm trong tập Train hoặc tập Val/Test để tránh hiện tượng rò rỉ dữ liệu (Data Leakage).
*   **Xử lý độ dài:** Sử dụng kỹ thuật **Sliding Window** với `max_len=512` và `stride=256` để xử lý các file code dài vượt quá giới hạn của model GraphCodeBERT.

In [ ]:
import json

dataset_metadata = {
    "dataset_name": "basic_cpp_ai_detection_dataset",
    "file_format": "JSON Lines (.jsonl)",
    "total_samples": 2850,
    "num_project_groups": 423,
    "target_language": "C++",
    "labels": {
        "0": "HUMAN",
        "1": "AI"
    },
    "class_distribution": {
        "HUMAN": "49.9%",
        "AI": "50.1%"
    },
    "schema": {
        "code": "Source code string",
        "label": "Text label (Human/AI)",
        "file_name": "Original filename containing author/repo info"
    },
    "preprocessing": {
        "grouping_logic": "user_repo (extracted from file_name)",
        "handling_long_sequences": "Sliding Window",
        "max_length": 512,
        "stride": 256
    },
    "validation_strategy": {
        "method": "Stratified Group K-Fold",
        "folds": 10,
        "leakage_prevention": "Group-based split by project_group"
    }
}

# Display the JSON
print(json.dumps(dataset_metadata, indent=4, ensure_ascii=False))

### 🚀 Tích hợp Metadata vào Quy trình Train & Save Model
Việc lưu metadata cùng với model giúp hệ thống tự động biết được các tham số quan trọng (như `stride` hay `threshold`) mà không cần cấu hình thủ công lại khi deploy.

In [ ]:
import os
import json

# Giả sử đây là bước sau khi bạn đã train xong và có best_threshold
# Chúng ta cập nhật các thông số thực tế từ quá trình train vào metadata
dataset_metadata["training_outputs"] = {
    "best_threshold": float(best_threshold) if 'best_threshold' in globals() else 0.5,
    "model_path_on_drive": DRIVE_SAVE_PATH if 'DRIVE_SAVE_PATH' in globals() else "./models",
    "python_version": "3.10",
    "transformers_version": "4.x"
}

# Tạo thư mục lưu trữ nếu chưa có
if not os.path.exists(DRIVE_SAVE_PATH):
    os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)

# Lưu metadata thành file JSON nằm cùng thư mục với Model weights
metadata_file_path = os.path.join(DRIVE_SAVE_PATH, "model_metadata.json")

with open(metadata_file_path, "w", encoding="utf-8") as f:
    json.dump(dataset_metadata, f, indent=4, ensure_ascii=False)

print(f"✅ Đã lưu metadata vào: {metadata_file_path}")

# --- Ví dụ cách sử dụng metadata khi load model để inference ---
# with open(metadata_file_path, 'r') as f:
#     meta = json.load(f)
#     current_threshold = meta['training_outputs']['best_threshold']
#     print(f"Model này cần sử dụng threshold: {current_threshold}")

### 🔍 Giải thích Quy trình Chọn Ngưỡng (Threshold Selection)

1.  **Dự đoán OOF (Out-of-Fold):** Trong mỗi fold, model dự đoán xác suất cho tập Validation (dữ liệu model chưa học). Các dự đoán này được tập hợp lại thành `oof_probs`.
2.  **Đường cong Precision-Recall:** Sử dụng `precision_recall_curve` để tính toán độ chính xác và độ triệu hồi ở nhiều ngưỡng khác nhau từ 0 đến 1.
3.  **Tối ưu F1-Score:** Tính toán F1-score cho mọi ngưỡng. Ngưỡng nào cho F1-score cao nhất sẽ được chọn làm `best_threshold`.
4.  **Ý nghĩa:** Ngưỡng này giúp cân bằng giữa việc bỏ sót code AI (False Negative) và việc nhận nhầm code người là AI (False Positive).

In [ ]:
# Hiển thị giá trị threshold cuối cùng đã tìm được
try:
    print(f"🎯 Ngưỡng quyết định (Threshold) cuối cùng: {best_threshold:.4f}")
    print(f"📈 F1-score tối ưu trên tập OOF: {f1_scores[best_idx]:.4f}")
except NameError:
    print("⚠️ Biến 'best_threshold' chưa được khởi tạo. Vui lòng chạy cell tìm threshold trước.")

In [ ]:
import matplotlib.pyplot as plt

def plot_threshold_optimization(y_true, y_probs, best_threshold):
    # Tính toán lại precision, recall, f1 để vẽ đồ thị
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    # Tránh chia cho 0
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

    plt.figure(figsize=(10, 6))

    # Vẽ các đường chỉ số
    plt.plot(thresholds, precisions[:-1], 'b--', label='Precision', alpha=0.7)
    plt.plot(thresholds, recalls[:-1], 'g--', label='Recall', alpha=0.7)
    plt.plot(thresholds, f1_scores[:-1], 'r-', label='F1-Score', linewidth=2)

    # Đánh dấu điểm tối ưu
    plt.axvline(x=best_threshold, color='black', linestyle=':', label=f'Best Threshold: {best_threshold:.4f}')
    plt.scatter([best_threshold], [max(f1_scores)], color='red')

    plt.title('Threshold Optimization via OOF Predictions')
    plt.xlabel('Threshold')
    plt.ylabel('Score')
    plt.legend(loc='lower left')
    plt.grid(True, alpha=0.3)
    plt.ylim([0, 1.05])
    plt.show()

# Chạy hàm vẽ đồ thị dựa trên dữ liệu OOF đã có
try:
    plot_threshold_optimization(oof_labels, oof_probs, best_threshold)
except NameError:
    print("⚠️ Không tìm thấy dữ liệu OOF. Hãy đảm bảo bạn đã chạy quá trình training và tìm threshold ở các cell trên.")

In [ ]:
import time
import numpy as np
from sklearn.metrics import precision_recall_curve

# Giả lập dữ liệu OOF lớn (ví dụ 10,000 mẫu dự đoán)
y_true_sim = np.random.randint(0, 2, 10000)
y_probs_sim = np.random.rand(10000)

start_time = time.time()

# Sklearn không duyệt từng cái một mà dùng thuật toán sắp xếp và quét hiệu quả
precisions, recalls, thresholds = precision_recall_curve(y_true_sim, y_probs_sim)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)
best_idx = np.argmax(f1_scores)

end_time = time.time()

print(f"⚡ Thời gian xử lý {len(thresholds)} ngưỡng khác nhau: {end_time - start_time:.4f} giây")
print(f"🎯 Ngưỡng tốt nhất tìm được: {thresholds[best_idx]:.4f}")

### 🛠️ Pipeline Quy trình Tối ưu hóa Threshold

Dưới đây là lộ trình từ con số dự đoán mặc định đến ngưỡng tối ưu cuối cùng:

1.  **Raw Predictions (Mặc định):**
    *   Model xuất ra xác suất (Probability) từ `0.0` đến `1.0`.
    *   Thông thường, các hệ thống dùng ngưỡng **0.5** (mặc định) để phân loại. Tuy nhiên, 0.5 chưa chắc là con số tốt nhất cho tập dữ liệu code C++ này.

2.  **Thu thập dữ liệu Out-of-Fold (OOF):**
    *   Trong quá trình train 10-fold, chúng ta thu thập tất cả xác suất dự đoán của tập Validation (những phần model chưa học).
    *   Kết quả thu được: `oof_probs` (danh sách xác suất) và `oof_labels` (nhãn thực tế).

3.  **Quét dải ngưỡng (Threshold Sweeping):**
    *   Sử dụng hàm `precision_recall_curve`, hệ thống sẽ thử nghiệm hàng ngàn ngưỡng khác nhau (ví dụ: 0.01, 0.02, ..., 0.99).

4.  **Tính toán chỉ số F1-Score:**
    *   Tại mỗi ngưỡng, tính **Precision** (độ chính xác) và **Recall** (độ triệu hồi).
    *   Tính **F1-Score** (trung bình điều hòa) để cân bằng cả hai.

5.  **Tìm điểm cực đại (Optimization):**
    *   Thuật toán tìm vị trí mà F1-Score đạt giá trị cao nhất.
    *   Kết quả tìm được là **0.6523**.

6.  **Kết luận:**
    *   Việc nâng ngưỡng từ 0.5 lên **0.6523** giúp giảm bớt các trường hợp 'báo động giả' (nhận nhầm code người là AI) trong khi vẫn giữ được độ nhạy cực cao với code AI.